# ResNet-50 Model Upload Pipeline: A Developer Template

[![Model](https://img.shields.io/badge/Model-ResNet--50-blue.svg)](https://arxiv.org/abs/1512.03385)
[![Dataset](https://img.shields.io/badge/Dataset-Stanford%20Cars-green.svg)](https://ai.stanford.edu/~jkrause/cars/car_dataset.html)

## Overview

This notebook is a self-contained template for downloading a computer vision model and uploading it to the Zededa EdgeAI platform using the `zededa-edgeai-sdk` CLI. It uses a **ResNet-50** model fine-tuned on the Stanford Cars dataset as a working example.

### Example Model Specifications
- **Architecture**: ResNet-50 (Deep Residual Network)
- **Task**: Fine-grained vehicle classification
- **Dataset**: Stanford Cars (196 vehicle categories)
- **Format**: ONNX

### Pipeline Steps
1. **Dependency Installation**: Install `huggingface-hub` and `zededa-edgeai-sdk`.
2. **Model Download**: Download the ONNX model from Hugging Face.
3. **CLI Setup**: Configure the service URL and verify the `zededa-eip` CLI installation.
4. **Authentication**: Log in and select your organization.
5. **Create a Local Provider**: Register a local file provider for your organization (one-time setup).
6. **Upload the Model**: Use the `zededa-eip` CLI to upload files and track the import job.

In [ ]:
import subprocess
import sys
from importlib.metadata import version as pkg_version, PackageNotFoundError
from packaging.version import Version

def check_package_installed(package_name, min_version=None):
    try:
        installed = pkg_version(package_name)
        if min_version and Version(installed) < Version(min_version):
            return False, installed
        return True, installed
    except PackageNotFoundError:
        return False, None

def install_package(package_spec):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package_spec])
        return True
    except subprocess.CalledProcessError:
        return False

required_packages = {
    "huggingface-hub": "0.35.0",
    "zededa-edgeai-sdk": "3.2.0",
}

print("🔍 Checking packages...")
missing_packages = [
    f"{pkg}>={ver}"
    for pkg, ver in required_packages.items()
    if not check_package_installed(pkg, ver)[0]
]

if missing_packages:
    print(f"📦 Installing {len(missing_packages)} packages...")
    failed = [p for p in missing_packages if not install_package(p)]
    if failed:
        print(f"⚠️  Failed to install: {', '.join(failed)}")
else:
    print("✅ All packages up-to-date")

import_failures = []
for module in ["huggingface_hub", "zededa_eip_client"]:
    try:
        __import__(module)
    except ImportError:
        import_failures.append(module)

if import_failures:
    print(f"❌ Import failures: {', '.join(import_failures)}")
else:
    print("✅ All critical imports verified")

In [ ]:
import os
from huggingface_hub import hf_hub_download
import shutil

repo_id = "zededa/resnet50-cars"
filename = "resnet50_cars_enhanced.onnx"

if os.path.exists(filename):
    local_model_path = os.path.abspath(filename)
    print(f"✅ Model exists: {filename} ({os.path.getsize(filename)/1024/1024:.1f} MB)")
else:
    print(f"📥 Downloading model from {repo_id}...")
    try:
        cached_path = hf_hub_download(repo_id=repo_id, filename=filename)
        local_model_path = os.path.abspath(filename)
        shutil.copy2(cached_path, local_model_path)
        print(f"✅ Downloaded: {filename} ({os.path.getsize(filename)/1024/1024:.1f} MB)")
    except Exception as e:
        print(f"❌ Download failed: {e}")
        local_model_path = None

## Step 3: CLI Setup

### 3.2 Configure the Service URL

Set `EDGEAI_SERVICE_URL` before running any CLI commands:

```bash
export EDGEAI_SERVICE_URL=https://edgeai.zededa.ai
```

To persist this across sessions, add the export to `~/.zshrc` (Zsh) or `~/.bashrc` (Bash).

> **Important:** The SDK default service URL in v3.0.0 is `https://studio.edgeai.zededa.dev`. Always set `EDGEAI_SERVICE_URL` explicitly to `https://edgeai.zededa.ai` before running any CLI commands for this POC.

In [ ]:
import os

DEFAULT_SERVICE_URL = "https://edgeai.zededa.ai"
service_url = input(f"Enter EdgeAI service URL (press Enter for default: {DEFAULT_SERVICE_URL}): ").strip() or DEFAULT_SERVICE_URL
os.environ["EDGEAI_SERVICE_URL"] = service_url
print(f"✅ EDGEAI_SERVICE_URL = {os.environ['EDGEAI_SERVICE_URL']}")

### 3.3 Verify the Installation

Confirm `zededa-eip` is installed and on your PATH. The second command lists all available sub-commands: `login`, `catalog`, `logout`, `set-catalog-context`, `external-providers`, `import-jobs`, `benchmarks`, and `device-pool`.

In [ ]:
!which zededa-eip
!zededa-eip

## Step 4: Authentication

The EdgeAI SDK authenticates using email and password credentials passed directly via CLI flags, without launching a browser.

### 4.1 Running the Login Command

```
zededa-eip login --email username@email.com --prompt-password
Password:
```

### 4.2 Selecting an Organization

If your account has access to multiple organizations, you will be prompted to select one:

```
Multiple organizations available. Please select one:
  1. zededa
  2. slb
  3. celona
Enter selection (1-3): 2
```

### 4.3 Successful Login

Upon successful login, the CLI launches an authenticated shell session. The selected organization is automatically set as `EDGEAI_CURRENT_ORG`, which the SDK uses to include the correct `X-Org-ID` header in all subsequent API requests:

```
Getting MinIO credentials...
Login successful. Launching interactive shell...
You are now in a shell with the following environment variables set:
  MLFLOW_TRACKING_TOKEN=eyJh...XE
  AWS_ACCESS_KEY_ID=AKIA...OE
  AWS_SECRET_ACCESS_KEY=2yLE...gP
  MLFLOW_S3_ENDPOINT_URL=https://s3.us-west-2.amazonaws.com
  MLFLOW_TRACKING_URI=https://edgeai.zededa.ai
  MINIO_BUCKET=edgeai-production-slb
  EDGEAI_CURRENT_ORG=slb
Type 'zededa-eip logout' to clear credentials or 'exit' to leave this shell.
```

> **Note:** These environment variables are active only within the authenticated shell session. They are cleared when you run `zededa-eip logout` or exit the shell.

In [ ]:
import getpass
import os
from zededa_eip_client.commands.login import execute_login

email = input("Email: ")
password = getpass.getpass("Password: ")

execute_login(email=email, password=password)

print(f"✅ Login successful. Organization: {os.environ.get('EDGEAI_CURRENT_ORG')}")

## Step 5: Creating a Local Provider

Before uploading an ML model, a provider must exist in the target organization. A provider defines the source mechanism for model files. For local file uploads, a local provider must be created once per organization.

### 5.1 Organization Context

After logging in, the SDK automatically sets `EDGEAI_CURRENT_ORG` to the selected organization. All subsequent commands automatically include the `X-Org-ID` header — no additional configuration is needed.

### 5.2 Create the Local Provider

Run the following command to create a local provider for your organization:

In [ ]:
!zededa-eip external-providers create \
  --name "local-provider" \
  --type local \
  --debug

## Step 6: Model Upload Workflow

This section describes how to upload a machine learning model to the Edge Intelligence Platform. The platform tracks each upload as an "import job", allowing you to monitor progress and verify completion.

### 6.1 Prepare Your Model Files

A typical model package includes:

| File | Purpose |
|------|---------|
| `model.onnx` (or `.pt`, `.tflite`) | The trained ML model in a supported format |
| `README.md` | Documentation describing the model, inputs, outputs, and usage |
| Sample image / test asset | Optional: a reference image or test file for validation |

> **Example:** For this POC, the model package is at `~/Downloads/resnet50_cars/` and contains: `README.md`, `resnet50_cars_enhanced.onnx`, and `zededa-resnet-50-cars.jpeg`

### 6.2 Upload the Model

```bash
zededa-eip import-jobs upload \
  --provider-name "local-provider" \
  --model-name "<your-model-name>" \
  --files <file1> <file2> <file3>
```

| Parameter | Description |
|-----------|-------------|
| `--provider-name` | Name of the provider to use. Use `"local-provider"` (created in Section 5). |
| `--model-name` | A unique name to identify the model (e.g., `resnet50_cars`). Reuse the same name to upload a new version — see Section 6.4. |
| `--files` | Space-separated list of file paths to upload. List each file individually. |

**Example Command**

In [ ]:
import subprocess
import re
import time

result = subprocess.run(
    [
        "zededa-eip", "import-jobs", "upload",
        "--provider-name", "local-provider",
        "--model-name", "resnet50_cars",
        "--files", local_model_path,
    ],
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(f"❌ Upload failed:\n{result.stderr}")
else:
    match = re.search(r"Import job ([a-f0-9\-]{36})", result.stdout)
    if not match:
        print("❌ Could not parse import job ID from output.")
    else:
        job_id = match.group(1)
        print(f"📋 Job ID: {job_id}")
        print("⏳ Polling job status...")
        while True:
            status_result = subprocess.run(
                ["zededa-eip", "import-jobs", "get", job_id],
                text=True,
                capture_output=True,
            )
            output = status_result.stdout.strip()
            print(output)
            if "completed" in output.lower():
                print("✅ Import job completed successfully.")
                break
            elif "failed" in output.lower():
                print("❌ Import job failed.")
                break
            else:
                time.sleep(5)

**Expected Output**

```
Import job 3446905c-8a91-4bb3-905a-46171e353aa2 created with status: pending
Use 'zededa-eip import-jobs get 3446905c-8a91-4bb3-905a-46171e353aa2' to check status
```

> **Important:** Save the import job ID returned in the output. Each upload command creates a new import job — check the status of a previous job before re-running to avoid duplicates.

### 6.3 Check Import Job Status

| Status | Meaning |
|--------|---------|
| `pending` | The job has been received and is queued for processing. |
| `running` | The platform is actively processing and importing the model files. |
| `completed` | The model has been successfully imported and is available on the platform. |
| `failed` | The import encountered an error. Review the job details for the error message. |

### 6.4 Uploading a New Version of a Model

To upload an updated version of an existing model, run the same upload command with the same model name but point `--files` to the new or updated files. The platform uses the model name to associate the new upload with the existing model and automatically registers it as a new version.

> **Important:** The model name must be exactly the same as the original upload. Using a different name will create a new, separate model entry rather than a new version of the existing one.

| Parameter | First Upload | New Version Upload |
|-----------|-------------|-------------------|
| `--provider-name` | `local-provider` | `local-provider` (unchanged) |
| `--model-name` | `resnet50_cars` | `resnet50_cars` — must be identical |
| `--files` | Original model files | Updated model files (can be new filenames or same filenames with updated content) |

**Command**

In [ ]:
import subprocess
import re
import time

new_model_path = input(f"Enter path to new version model file (press Enter to use current: {local_model_path}): ").strip() or local_model_path

result = subprocess.run(
    [
        "zededa-eip", "import-jobs", "upload",
        "--provider-name", "local-provider",
        "--model-name", "resnet50_cars",
        "--files", new_model_path,
    ],
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(f"❌ Upload failed:\n{result.stderr}")
else:
    match = re.search(r"Import job ([a-f0-9\-]{36})", result.stdout)
    if not match:
        print("❌ Could not parse import job ID from output.")
    else:
        job_id = match.group(1)
        print(f"📋 Job ID: {job_id}")
        print("⏳ Polling job status...")
        while True:
            status_result = subprocess.run(
                ["zededa-eip", "import-jobs", "get", job_id],
                text=True,
                capture_output=True,
            )
            output = status_result.stdout.strip()
            print(output)
            if "completed" in output.lower():
                print("✅ Import job completed successfully.")
                break
            elif "failed" in output.lower():
                print("❌ Import job failed.")
                break
            else:
                time.sleep(5)

**Expected Output**

```
Import job 7f3a1c2d-0011-4abc-b567-89def0123456 created with status: pending
Use 'zededa-eip import-jobs get 7f3a1c2d-0011-4abc-b567-89def0123456' to check status
```

A new import job ID is returned for the version upload. Track it independently to confirm the new version has been processed successfully.

> **Best practice:** Always verify the previous import job has reached `completed` status before uploading a new version, to avoid concurrent processing conflicts.